In [0]:
# =============================================================================
# Notebook  : 00_master_run_report
# Location  : /HGI-Lakehouse-Pipeline/00_Orchestration/00_master_run_report
# Purpose   : Final task in the master job. Prints a comprehensive end-to-end
#             summary of the entire pipeline run:
#               - Files detected per source
#               - Records before/after per layer per object
#               - Net new records written
#               - Silver table sizes
#               - All spec DQ gate results
#               - Run duration and cost estimate
#               - History of last 10 full pipeline runs
# =============================================================================

In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_002")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
try:
    tenant_id = int(customer_id.split("_")[1])
except Exception:
    tenant_id = -1

from datetime import datetime
run_start = datetime.utcnow()

print("=" * 70)
print("  🏁  MASTER PIPELINE RUN REPORT")
print("=" * 70)
print(f"  Customer    : {customer_id}  (tenant={tenant_id})")
print(f"  Report time : {run_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")


In [0]:
# =============================================================================
# SECTION 1: CURRENT RUN SUMMARY FROM LOG TABLE
# =============================================================================

# CELL 3 ── What happened this run
print(f"\n{'─'*70}")
print("  SECTION 1: THIS RUN — What Was Processed")
print(f"{'─'*70}")

try:
    # Get the most recent batch of runs (same approximate timestamp = same pipeline run)
    this_run = spark.sql(f"""
        WITH latest_run AS (
            SELECT MAX(run_timestamp) AS latest
            FROM {meta_catalog}.pipeline_run_log
            WHERE customer_id = '{customer_id}'
        )
        SELECT
            layer,
            source_system,
            object_name,
            status,
            files_found,
            records_before,
            records_after,
            records_new,
            run_timestamp
        FROM {meta_catalog}.pipeline_run_log l, latest_run lr
        WHERE l.customer_id = '{customer_id}'
          AND l.run_timestamp >= TIMESTAMPADD(MINUTE, -30, lr.latest)
        ORDER BY layer, source_system, object_name
    """).collect()

    if this_run:
        # Group by layer
        layers_order = ["landing", "bronze", "silver_cdf", "map"]
        by_layer = {}
        for row in this_run:
            by_layer.setdefault(row["layer"], []).append(row)

        total_files_sf  = 0
        total_files_bq  = 0
        total_records_new = 0

        for layer in layers_order:
            rows = by_layer.get(layer, [])
            if not rows:
                continue
            print(f"\n  Layer: {layer.upper()}")
            print(f"  {'Source/Object':<30} {'Status':<12} {'Files':<8} {'Before':>10} {'After':>10} {'New':>10}")
            print(f"  {'-'*28} {'-'*10} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")
            for row in rows:
                icon = {"success":"✅","failed":"❌","skipped":"⏭","no_new_data":"⏸"}.get(row["status"],"ℹ")
                src  = f"{row['source_system']}/{row['object_name']}"
                print(f"  {src:<30} {icon}{row['status']:<10} {row['files_found']:>7,} "
                      f"{row['records_before']:>10,} {row['records_after']:>10,} "
                      f"{row['records_new']:>10,}")
                if row["source_system"] == "bigquery":
                    total_files_bq += (row["files_found"] or 0)
                else:
                    total_files_sf += (row["files_found"] or 0)
                total_records_new += (row["records_new"] or 0)

        print(f"\n  ── Run Totals ──")
        print(f"  SF files processed   : {total_files_sf:,}")
        print(f"  BQ files processed   : {total_files_bq:,}")
        print(f"  Total new records    : {total_records_new:,}")
    else:
        print("  No log entries found for this run yet.")
except Exception as e:
    print(f"  Log table not available: {e}")

In [0]:
# =============================================================================
# SECTION 2: CURRENT STATE OF ALL TABLES
# =============================================================================

# CELL 4 ── Bronze table sizes
print(f"\n{'─'*70}")
print("  SECTION 2: BRONZE TABLE STATE")
print(f"{'─'*70}")
print(f"  Table (bronze.{customer_id}.*)      {'Rows':>12}")
print(f"  {'─'*38} {'─'*12}")

bronze_total = 0
for obj, tbl in BRONZE_TABLES.items():
    full = f"{bronze_catalog}.{customer_id}.{tbl}"
    try:
        n = spark.table(full).count()
        bronze_total += n
        print(f"  {tbl:<40} {n:>12,}")
    except Exception:
        print(f"  {tbl:<40} {'NOT FOUND':>12}")
print(f"  {'─'*38} {'─'*12}")
print(f"  {'TOTAL':<40} {bronze_total:>12,}")



In [0]:
# CELL 5 ── Silver table sizes (this tenant only)
print(f"\n{'─'*70}")
print(f"  SECTION 3: SILVER TABLE STATE (tenant={tenant_id} only)")
print(f"{'─'*70}")
print(f"  Table (hgi.silver.*)                {'Rows':>12}")
print(f"  {'─'*38} {'─'*12}")

silver_base = ["accounts","contacts","opportunities","tasks",
               "campaigns","campaign_members","users","events"]
silver_total = 0
for tbl in silver_base:
    try:
        n = spark.sql(
            f"SELECT COUNT(*) FROM {sv}.{tbl} WHERE tenant={tenant_id}"
        ).collect()[0][0]
        silver_total += n
        print(f"  {tbl:<40} {n:>12,}")
    except Exception:
        print(f"  {tbl:<40} {'NOT FOUND':>12}")
print(f"  {'─'*38} {'─'*12}")
print(f"  {'TOTAL (this tenant)':<40} {silver_total:>12,}")


In [0]:
# CELL 6 ── Map output table sizes
print(f"\n{'─'*70}")
print("  SECTION 4: MAP LAYER TABLE STATE")
print(f"{'─'*70}")
print(f"  Table (hgi.silver.*)                {'Rows':>12}  Description")
print(f"  {'─'*38} {'─'*12}  {'─'*25}")

map_tables = [
    ("contacts_all",             "Unified contacts view"),
    ("accounts_all",             "Unified accounts view"),
    ("crm_events",               "SF Tasks → meta_events"),
    ("mapped_events",            "All events, contactid resolved"),
    ("contacts_to_accounts",     "3-phase contact-account links"),
    ("accounts_attributes",      "Account is_paying/mrr"),
    ("contacts_attributes",      "Contact is_paying/mrr"),
    ("email_events_mapped",      "Email×event×day aggregation"),
    ("domain_events_mapped",     "Domain×event×day aggregation"),
    ("mk_account_events_mapped", "Account event rollup"),
    ("email_audience",           "Audience segments (stub)"),
    ("email_model_conversion",   "Conversion tracking (stub)"),
]
for tbl, desc in map_tables:
    try:
        n = spark.table(f"{sv}.{tbl}").count()
        print(f"  {tbl:<40} {n:>12,}  {desc}")
    except Exception:
        print(f"  {tbl:<40} {'─':>12}  {desc} (not built yet)")

In [0]:
# =============================================================================
# SECTION 5: SPEC DQ GATES
# =============================================================================

# CELL 7 ── All 5 spec quality gates
print(f"\n{'─'*70}")
print("  SECTION 5: SPEC DQ GATES (from client documents)")
print(f"{'─'*70}")

dq_gates = []

# Gate 1: meta_event 100% non-null
try:
    total_ev = spark.table(f"{sv}.mapped_events").count()
    null_meta = spark.sql(f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE meta_event IS NULL").collect()[0][0]
    pct = round(100*(total_ev-null_meta)/max(total_ev,1),1)
    pass_fail = "✅ PASS" if null_meta == 0 else "❌ FAIL"
    dq_gates.append(("meta_event 100% non-null", pct, 100.0, "%", pass_fail))
except Exception:
    dq_gates.append(("meta_event 100% non-null", "N/A", 100.0, "%", "⏸ SKIP"))

# Gate 2: events contact resolution ≥70%
try:
    total_ev = spark.table(f"{sv}.mapped_events").count()
    w_contact = spark.sql(f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE contactid IS NOT NULL").collect()[0][0]
    pct = round(100*w_contact/max(total_ev,1),1)
    threshold = DQ_THRESHOLDS["event_contact_resolution_pct"]
    pass_fail = f"✅ PASS" if pct >= threshold else f"⚠  BELOW {threshold}%"
    dq_gates.append((f"events contact resolution ≥{threshold}%", pct, threshold, "%", pass_fail))
except Exception:
    dq_gates.append(("events contact resolution", "N/A", 70.0, "%", "⏸ SKIP"))

# Gate 3: contacts_to_accounts linkage ≥80%
try:
    total_c = spark.table(f"{sv}.contacts_all").count()
    linked  = spark.table(f"{sv}.contacts_to_accounts").count()
    pct = round(100*linked/max(total_c,1),1)
    threshold = DQ_THRESHOLDS["c2a_linkage_pct"]
    pass_fail = "✅ PASS" if pct >= threshold else f"⚠  BELOW {threshold}%"
    dq_gates.append((f"c2a linkage ≥{threshold}%", pct, threshold, "%", pass_fail))
except Exception:
    dq_gates.append(("contacts_to_accounts linkage", "N/A", 80.0, "%", "⏸ SKIP"))

# Gate 4: c2a orphan contact_ids = 0
try:
    orphans = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.contacts_all c ON c2a.contact_id = c.id
        WHERE c.id IS NULL
    """).collect()[0][0]
    pass_fail = "✅ PASS" if orphans == 0 else f"❌ {orphans} orphans"
    dq_gates.append(("c2a orphan contact_ids = 0", orphans, 0, "count", pass_fail))
except Exception:
    dq_gates.append(("c2a orphan contact_ids", "N/A", 0, "count", "⏸ SKIP"))

# Gate 5: accounts_attributes covers all accounts
try:
    uncovered = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all a
        LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
        WHERE aa.account_id IS NULL
    """).collect()[0][0]
    pass_fail = "✅ PASS" if uncovered == 0 else f"❌ {uncovered} uncovered"
    dq_gates.append(("accounts_attributes coverage = 0", uncovered, 0, "count", pass_fail))
except Exception:
    dq_gates.append(("accounts_attributes coverage", "N/A", 0, "count", "⏸ SKIP"))

for gate_name, actual, target, unit, result in dq_gates:
    print(f"  {result}  {gate_name:<45}  actual={actual}{unit}")

In [0]:
# =============================================================================
# SECTION 6: PIPELINE RUN HISTORY
# =============================================================================

# CELL 8 ── Last 10 complete pipeline runs
print(f"\n{'─'*70}")
print(f"  SECTION 6: PIPELINE RUN HISTORY (last 10 runs for {customer_id})")
print(f"{'─'*70}")

try:
    spark.sql(f"""
        SELECT
            DATE_FORMAT(run_timestamp, 'yyyy-MM-dd HH:mm') AS run_time,
            layer,
            source_system,
            object_name,
            status,
            files_found        AS files,
            records_before     AS before,
            records_new        AS new_recs
        FROM {meta_catalog}.pipeline_run_log
        WHERE customer_id = '{customer_id}'
        ORDER BY run_timestamp DESC
        LIMIT 20
    """).show(20, truncate=False)
except Exception as e:
    print(f"  Could not read run history: {e}")

print(f"\n{'='*70}")
print(f"  Master run report complete: {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*70}")

dbutils.notebook.exit("report_complete")
